# Demo: QMLBiClase — Clasificación binaria (Iris subset)

Notebook demostrativo que crea la instancia QMLBiClase, ejecuta cross-validation
y muestra métricas básicas. Ajusta `epochs` y `shots` para que el demo sea rápido.

In [ ]:
# Asegurar que el paquete 'src' está en sys.path cuando se ejecuta desde examples/qml/
import sys
from pathlib import Path
nb_path = Path.cwd()
repo_root = nb_path
# buscar carpeta 'src' hacia arriba
for _ in range(6):
    if (repo_root / 'src').exists():
        break
    repo_root = repo_root.parent
src_path = repo_root / 'src'
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))
else:
    # fallback: añadir repo root
    sys.path.insert(0, str(repo_root.resolve()))
print('Using import path:', sys.path[0])

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np

from quantum_metric.models.qml_biclase import QMLBiClase
from quantum_metric.validation.cross_validation import cross_validate_biclase

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

iris = load_iris()
X = iris.data
y = iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_encoded = LabelEncoder().fit_transform(y)

# Tomar solo clases 0 y 1 para un ejemplo binario
mask = y_encoded < 2
X_bin = X_scaled[mask]
y_bin = y_encoded[mask]
print('Samples:', X_bin.shape)


In [ ]:
# Crear instancia ligera para demo
model = QMLBiClase(
    codigo='gray',
    noise=0.0,
    backend=None,
    noise_model=None,
    result='probs',
    shots=1024,
    lr=0.1,
    epochs=2,    # pocas epochs para demo
    optimize=False,
    rng=0
)
print('Model created:', model)


In [ ]:
# Ejecutar cross-validation (usa la función del paquete). Ajusta kwargs si tu modelo los requiere.
res = cross_validate_biclase(model, X_bin, y_bin, k_folds=3, epochs=2, lr=0.1, rng=0, optimize=False)
print('\nResultado de cross-validate:')
print(res)


In [ ]:
# Alternativa: usar fit_validation del propio modelo y luego predict
try:
    model.fit_validation(X_bin, y_bin)
    preds = model.predict()
    print('\nPredicciones (primeras 20):', preds[:20])
except Exception as e:
    print('fit_validation/predict falló:', e)


Notas:
- Si tu entorno no tiene PennyLane o backend compatible, las celdas de fit/predict pueden fallar.
- Reduce `epochs` y `shots` para pruebas rápidas. Para entrenamientos reales incrementa ambos.
- Revisa `src/quantum_metric/models/qml_biclase.py` y `validation/cross_validation.py` si cambias firmas.